In [1]:
from pathlib import Path
from rich import print

### Load configuration


In [ ]:
from w2t_bkin import config, sync, behavior, session, bpod, events, ttl

### Ingest and verify


In [3]:
session_dir = Path("data/raw/Session-000001")
nwbfile = session.create_nwb_file(session_dir / "session.toml")
nwbfile

/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/pynwb/file.py:492: UserWarning: Date is missing timezone information. Updating to local timezone.
  args_to_set['session_start_time'] = _add_missing_timezone(session_start_time)
/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/pynwb/file.py:154: UserWarning: Date is missing timezone information. Updating to local timezone.
  args_to_set['date_of_birth'] = _add_missing_timezone(date_of_birth)


root pynwb.file.NWBFile at 0x127295499732880
Fields:
  data_collection: Data collected using Bpod behavioral control system.
Video recorded at 30 fps using synchronized cameras.
Pose estimated using DeepLabCut model trained on lab-specific data.

  devices: {
    bpod <class 'pynwb.device.Device'>,
    camera_0 <class 'pynwb.device.Device'>,
    camera_1 <class 'pynwb.device.Device'>
  }
  experiment_description: Water-to-target behavioral task with simultaneous video tracking.
Animal navigates to target location while pose is tracked using
DeepLabCut and synchronized with Bpod behavioral events.

  experimenter: ['John Doe' 'Jane Smith']
  file_create_date: [datetime.datetime(2025, 11, 25, 19, 18, 6, 431390, tzinfo=tzlocal())]
  identifier: Session-000001
  institution: Neural Dynamics Laboratory, University Example
  keywords: ['behavior' 'pose tracking' 'synchronization' 'water reward' 'navigation']
  lab: Larkum Lab
  notes: Session conducted with standard lighting conditions.
Animal showed normal behavior throughout the session.
All equipment functioning properly.

  pharmacology: None
  protocol: IACUC-2025-001
  related_publications: []
  session_description: Behavioral training session with pose tracking and camera synchronization
  session_id: 000001
  session_start_time: 2025-01-15 14:30:00+01:00
  slices: N/A - in vivo experiment
  source_script: /home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/ipykernel_launcher.py
  source_script_file_name: ipykernel_launcher.py
  stimulus_notes: Visual target displayed on LED screen at variable locations
  subject: subject pynwb.file.Subject at 0x127295305967184
Fields:
  age: P84D
  age__reference: birth
  date_of_birth: 2024-10-23 00:00:00+02:00
  description: Adult male C57BL/6J mouse, 12 weeks old
  genotype: C57BL/6J wild-type
  sex: M
  species: Mus musculus
  strain: C57BL/6J
  subject_id: M001
  weight: 0.025 kg

  surgery: Cranial window implant surgery performed 2 weeks prior to recording.
No complications observed. Animal recovered fully.

  timestamps_reference_time: 2025-01-15 14:30:00+01:00
  virus: None
  was_generated_by: ['w2t_bkin==0.0.3' 'pynwb==3.1.2' 'hdmf==4.1.2' 'deeplabcut==2.3.11'
 'facemap==1.0.8' 'scipy==1.15.3' 'numpy==1.26.4' 'pandas==2.3.3'
 'torch==2.9.1']

### Import events data (TTL signals)

In [ ]:
ttl_patterns = {"ttl_camera": "TTLs/cam*.txt"}
ttl_pulses = ttl.get_ttl_pulses(session_dir, ttl_patterns)

In [ ]:
events.extract_ttl_table(ttl_pulses)

### Parse Bpod behavioral data


In [4]:
bpod_data = bpod.parse_bpod(session_dir=session_dir, pattern="Bpod/*.mat", order="name_asc", continuous_time=False)
bpod_data

{'__header__': b'MATLAB 5.0 MAT-file Platform: posix, Created on: Fri Nov 14 13:29:31 2025',
 '__version__': '1.0',
 '__globals__': [],
 'SessionData': {'Analog': <scipy.io.matlab._mio5_params.mat_struct at 0x73c63f7e4df0>,
  'Info': <scipy.io.matlab._mio5_params.mat_struct at 0x73c63f7e5f60>,
  'SettingsFile': <scipy.io.matlab._mio5_params.mat_struct at 0x73c63f7e5b10>,
  'nTrials': 3,
  'RawEvents': {'Trial': [<scipy.io.matlab._mio5_params.mat_struct at 0x73c63f7e5900>,
    {'States': <scipy.io.matlab._mio5_params.mat_struct at 0x73c63f87a170>,
     'Events': <scipy.io.matlab._mio5_params.mat_struct at 0x73c63f87a140>}]},
  'RawData': <scipy.io.matlab._mio5_params.mat_struct at 0x73c63f7e6080>,
  'TrialStartTimestamp': [0.1438, 11.3443, 22.0374],
  'TrialEndTimestamp': [11.3157, 22.0135, 30.4916],
  'TrialSettings': [<scipy.io.matlab._mio5_params.mat_struct at 0x73c63f879c30>,
  'TrialTypes': [2, 2, 1]}}

In [5]:
behavior.extract_task(bpod_data)

,event_name
id,
0,BNC1High
1,BNC1Low
2,Flex1Trig1
3,Flex1Trig2
,state_name
id,
0,A2L_Audio
1,Airpuff
2,HIT


In [6]:
behavior.extract_task_recording(bpod_data)

/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'event_type' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()
/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'state_type' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()
/home/borja/w2t-bkin/.venv/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'action_type' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()


,timestamp,event_type,value
id,,,
0,0.1439,3,Flex1Trig2
1,8.1359,3,Flex1Trig2
2,8.2839,3,Flex1Trig2
3,4.5179,2,Flex1Trig1
,start_time,stop_time,state_type
id,,,
0,0.1438,7.3833,3
1,7.3833,7.8833,0
2,7.8833,8.1333,1


In [7]:
behavior.extract_trials_table(bpod_data)

,start_time,stop_time,states,events,actions
id,,,,,
0,0.1438,11.3157,"[0, 1, 2, 3, 4, 5, 6, 7, 8]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]","[0, 1, 2]"
1,11.3443,22.0135,"[9, 10, 11, 12, 13, 14, 15, 16, 17]","[22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65]","[3, 4, 5]"
2,22.0374,30.4916,"[18, 19, 20]","[66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93]",[6]


### Import pose data


### Import facemap data


### Transcode videos for better storage


### Synchronize to reference timebase


In [8]:
sync_data = sync.parse_ttl_data(ttldata_dir, pattern="*.tsq", order

SyntaxError: incomplete input (1341220758.py, line 1)

In [ ]:
ttl_patterns = {ttl.id: ttl.paths for ttl in session.TTLs}
ttl_pulses = sync.get_ttl_pulses(ttl_patterns, session.session_dir)
trial_offsets, warnings = sync.align_bpod_trials_to_ttl(
    trial_type_configs=session.bpod.trial_types,
    bpod_data=bpod_data,
    ttl_pulses=ttl_pulses,
)

In [ ]:
behavior.extract_task(bpod_data)

In [ ]:
behavior.extract_task_recording(bpod_data, trial_offsets)

In [ ]:
behavior.extract_trials_table(bpod_data, trial_offsets)

### Assemble NWB


In [ ]:
output_dir = Path("data/external/Session-000001")
provenance = {
    "config_hash": "abc123",
    "session_hash": "def456",
    "software": {"name": "w2t_bkin", "version": "0.1.0"},
    "timebase": {"source": "nominal_rate"},
}
nwb_path = nwb.assemble_nwb(manifest, cfg, provenance, output_dir)

### Validate and QC (Phase 5 - not yet implemented)
